In [26]:
##### Calculates final capital and labor intensities using final production and capital/labor rasters (after re-scaling)

import os
import pandas as pd
import geopandas as gpd
import rioxarray as rio
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
from glob import glob
import rasterio
from rasterio.warp import reproject, Resampling
from matplotlib.colors import BoundaryNorm
import matplotlib.colors as mcolors
from pyproj import Transformer
from pathlib import Path

In [27]:
##### Load data

# Get the current working directory
cd = Path.cwd().parent.parent 

# Import data
capital = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD.tif")
capital_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_p10.tif")
capital_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_capital_USD_p90.tif")

labor = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs.tif")
labor_p10 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_p10.tif")
labor_p90 = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/rescaled_jobs_p90.tif")

production = rio.open_rasterio(f"{cd}/Results/Raster_model/reprojected/total_production_tonnes_2020.tif")

In [28]:
##### Zero out small capital/labor values before computing intensities
# avoids extremely small intensities 

crs = capital.rio.crs 

capital = xr.where(capital >= 1, capital, 0)
capital_p10 = xr.where(capital_p10 >= 1, capital_p10, 0)
capital_p90 = xr.where(capital_p90 >= 1, capital_p90, 0)

labor = xr.where(labor >= 1, labor, 0)
labor_p10 = xr.where(labor_p10 >= 1, labor_p10, 0)
labor_p90 = xr.where(labor_p90 >= 1, labor_p90, 0)

##### If production < 1 and either capital or labor is 0, set both to NaN
def nan_where_either_zero(cap, lab, production):
    mask = (production < 1) & ((cap == 0) | (lab == 0))
    cap_out = xr.where(mask, np.nan, cap)
    lab_out = xr.where(mask, np.nan, lab)
    return cap_out, lab_out

capital, labor = nan_where_either_zero(capital, labor, production)
capital_p10, labor_p10 = nan_where_either_zero(capital_p10, labor_p10, production)
capital_p90, labor_p90 = nan_where_either_zero(capital_p90, labor_p90, production)

In [29]:
##### Calculate and save intensity rasters (central, p10, p90)
def compute_and_save_intensity(numerator, production_masked, crs, scale, out_path):
    intensity = (numerator / production_masked) * scale
    intensity = intensity.where(np.isfinite(intensity))
    intensity = intensity.rio.write_nodata(np.nan)
    intensity = intensity.rio.write_crs(crs)
    intensity.rio.to_raster(out_path, dtype="float32", compress="LZW")
    return intensity

# mask production to get rid of very small production blow up in intensities 
production_masked = production.where(production >= 0.1)

out_dir = f"{cd}/Results/Raster_model"

variants = {
    "": {"capital": capital, "labor": labor},
    "_p10": {"capital": capital_p10, "labor": labor_p10},
    "_p90": {"capital": capital_p90, "labor": labor_p90},
}

results = {}

for suffix, data in variants.items():
    results[f"capital{suffix}"] = compute_and_save_intensity(
        numerator=data["capital"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/capital_intensity_USD_per_tonne{suffix}.tif",
    )
    results[f"labor{suffix}"] = compute_and_save_intensity(
        numerator=data["labor"],
        production_masked=production_masked,
        crs=crs,
        scale=1,
        out_path=f"{out_dir}/labor_intensity_jobs_per_tonne{suffix}.tif",
    )